# Measurement of W + charm production using CMS Open Data:

This notebook involves the study of **W+charm** using CMS Open Data with W+Jets in final state. 

## Key Idea: 

This study is of the associated production of a W boson and a charm(c) quark at the LHC. This provides direct access to strange quark content of the proton at the W-boson mass energy scale. This sensitivity is due to dominance of $\bar{q}g \rightarrow W^{+} + \bar{c}$ and $qg \rightarrow W^{-} + c$ contibutions at leading order.

## Sample Used:

- **Simulated Dataset:** [Simulated dataset WJetsToLNu_TuneCP5_13TeV-madgraphMLM-pythia8 in NANOAODSIM format for 2016 collision data](https://opendata.cern.ch/record/69747). 80958227 events. 68 files. 73.9 GiB in total. (Here we take only partially available file - 2006990 events).

- **Collision Dataset:** TBA

## 1. Setup:

We use `coffea` for event processing, `awkward-array` and `dask` for jagged array handling, and `mplhep` to produce publication-ready CMS plots. All of them are available in the **Scikit-HEP** toolkit.

We also import `hist_functions` which contains helper plotter functions `make_hist`, `make_diff_hist`, `make_overlay_hist` and `os_ss_hist`.

In [1]:
import hist as h
import dask_awkward as dak
import numpy as np
import hist_functions
from hist_functions import make_hist, make_diff_hist, make_overlay_hist, os_ss_hist
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import vector
vector.register_awkward()

NanoAODSchema.warn_missing_crossrefs = False

## 2. Loading the data:
We load the WJetsToLNu sample of simulated $W\rightarrow \ell + \nu$. `dask` is used to lazily read the file to save resources and only read from disk whenever necessary.

In [2]:
# fname = "../../datasets/WJets_ToLNu_NANOAODSIM.root"
fname = "../../datasets/nano106X_on_mini106X_2017_mc_NANOAOD_W1Jets_to_LNu_250K.root"
print(f"Loading file: {fname}")
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset":"WJets"},
    mode="dask"
).events()
events

Loading file: ../../datasets/nano106X_on_mini106X_2017_mc_NANOAOD_W1Jets_to_LNu_250K.root


dask.awkward<from-uproot, npartitions=1>

## 3. Event Selection Strategy:

### Analysis Strategy:

The Event selection strategy is as follows:
- **W-Boson selection:** Indetification of W-boson is done via its leptonic decay channel ($W \rightarrow \mu + \nu$) where we select high-$p_{T}$, isolated "tight" muons and missing transverse energy ($p_{T}^{miss}$).
- **Charm selection:** Identification of charm is done by looking for non-isolated "soft" muons from semi-leptonic decays of C-Hadron ($c \rightarrow s\mu\nu$).
- **OS-SS Subtraction:** The muon associated with the W-boson and the muon associated with the c-jet should have **opposite sign (OS)**. Here background processes like gluon splitting processes, Drell-Yan or $t\bar{t}$ are charge symmetric. By subtracting **same sign (SS)** events from OS events, we statistically reduce the background to obtain the signal.  

### Some Important Notes:

- In the $W+charm$ process at leading-order (LO), the electric charges of the $W$ boson and the $charm$ quark have opposite sign(OS).

- **Background:** Gluon splitting $q\bar{q} \rightarrow W + g \rightarrow W + c\bar{c}$ process produces both $c$ and $\bar{c}$; so the OS and SS events are both equally likely.  

- Detectors often reconstruct/tag only one c-jet candidate out of the two in the gluon splitting process.

- With only one picked c-jet, gluon splitting events becomes OS/SS-symmetric. That OS and SS yields from this background is equally likely.

- Why only one c-jet is tagged? The reason is limited acceptance, reconstruction efficiency and c-tagging efficiency meaning often times the detector can only reconstruct one of the either c-jet candidate from $g \rightarrow c\bar{c}$. The untagged c-jet candidate might be too soft or too forward to pass trigger or fail tagging.

- In $W + g \rightarrow g \rightarrow c \bar{c}$ background, if we keep a fixed $W$ charge. Let $S$ be true signal, $B$ a symmetric background: $N_{OS} = S + \dfrac{1}{2}B$ and $N_{SS} = 0 + \dfrac{1}{2}B$. Then subtract $N_{OS} - N_{SS}$, this cancels background and only signal remains. Here $N_{OS}$ and $N_{SS}$ are yields of opposite-sign and same-sign events.

## 4. Pre-selection and Initial Sanity Checks:

Here, we will select make object collections of _Muons_, _MET_ and _Jets_ that we will use later on in our analysis and also make some initial sanity checks and plots.

In [3]:
print("Defining Physics Objects...")

cutflow = {}

# Collections of physics objects that we want for this analysis:
Muon = events.Muon
MET = events.MET
Jet = events.Jet

print("Defining physics objects ... Successful!")
print("--------------------------------------------")
# Count the number of events in the dataset:
n_events = dak.num(events, axis=0)
print("Total number of events in the dataset:", n_events.compute())

# --- CUTFLOW 1 --- #
cutflow["1. Total Events"] = n_events

# Count the number of muons in the dataset:
nMuons = dak.num(Muon, axis=1)
total_Muons = dak.sum(nMuons)
print("Total number of muons in the dataset:", total_Muons.compute())

# Count the number of jets in the dataset:
nJets = dak.num(Jet, axis=1)
total_Jets = dak.sum(nJets)
print("Total number of jets in the dataset:", total_Jets.compute())

Defining Physics Objects...
Defining physics objects ... Successful!
--------------------------------------------
Total number of events in the dataset: 250000
Total number of muons in the dataset: 94049
Total number of jets in the dataset: 940357


## 5. W-Boson Selection:

The following cuts are applied to obtain a high-$p_{T}$ and isolated "tight muon" with the trigger `HLT_IsoMu24`:

- Muon $p_{T}$ > 25 GeV. (high-$p_{T}$)
- Muon $\eta$ < 2.4. 
- Muon `pfRelIso04_all` < 0.15. (isolated)
- Muon `tightID` == True.

We additionally also put a cut on the Transverse Mass ($M_{T}$ > 50 GeV) of Muon+MET system to ensure the selected candidate is associated with a W-Boson.

In [4]:
print("Defining W Selection ...")
######################################
## W-Boson -> Mu Nu Channel Selection:
######################################

# W-Boson selection cuts:
W_MuMask = (
    (Muon.pt > 25) &
    (abs(Muon.eta) < 2.4) &
    (Muon.tightId == 1) &
    (Muon.pfRelIso04_all < 0.15)
    )

# Transverse mass calculation and cut:
Mt = np.sqrt(2 * Muon.pt * MET.pt * (1 - np.cos(Muon.phi - MET.phi)))
Mt_Cut = (Mt > 50)
W_MuMask = W_MuMask & Mt_Cut
W_Muons = Muon[W_MuMask]

# Select events with W muons:
nMu = dak.num(W_Muons, axis=1)
evt_has_WMu = (nMu >= 1)

W_Muons = W_Muons[evt_has_WMu] 

nMu = dak.num(W_Muons, axis=1)
nMu = dak.sum(nMu)
cutflow["2. W-Muons after cuts"] = nMu
print("Total number of W muons:", nMu.compute())

Defining W Selection ...
Total number of W muons: 31118


## 6. Charm Selection:

In [5]:
print("Defining charm Selection ...")
###############################################
## C-jet -> Mu X Channel Selection (soft muon):
###############################################

cMuPassCuts = (
    (Muon.pt < 25) &
    (abs(Muon.eta) < 2.4) &
    (Muon.pfRelIso04_all > 0.2) & 
    (Muon.tightId == 1)
)
# Impact parameter significance cut (3D):
dxy_err_ok = abs(Muon.dxyErr) > 0 # just a safe check to avoid division by zero
sip2d = abs(Muon.dxy) / Muon.dxyErr 
sip3d = Muon.sip3d
ip2dSigMask = (sip2d > 2)
ip3dSigMask = (abs(sip3d) > 2)

soft_muon = Muon[cMuPassCuts & ip3dSigMask]

# Print the number of soft muons passing the selection:
nSoftMuons = dak.num(soft_muon, axis=1)
total_SoftMuons = dak.sum(nSoftMuons)
print("Total number of soft muons:", total_SoftMuons.compute())

###############################################
## dR matching (soft muon <-> jets)
## - builds all muon-jet pairs per event
## - picks nearest jet per muon
## - requires min dR < dr_cut
###############################################

# Jets eligible for matching (apply your jet kinematic cuts here)
jets = Jet[
    (Jet.pt > 30) &
    (abs(Jet.eta) < 2.4)
]

# Summary of jet availability for matching:
njets = dak.num(jets, axis=1)
total_Jets = dak.sum(njets)
print("Total number of jets eligible for matching:", total_Jets.compute())

# Pairwise muon-jet combinations within each event: (evt, nMu, nJet)
dr_cut = 0.4
pairs = dak.cartesian({"mu": soft_muon, "jet": jets}, nested=True)

# Calculate deltaR for each muon-jet pair 
dr_pairs = pairs["mu"].delta_r(pairs["jet"])

# Nearest jet per muon (handles events with 0 jets by returning None)
# axis=0 (nEvent), axis=1 (nMu), axis=2 (nJet)
min_dr = dak.min(dr_pairs, axis=2) # Gets the nearest jet distance for each muon
best_jet_idx = dak.argmin(dr_pairs, axis=2) # Gets the index of the nearest jet for each muon 
min_dr_filled = dak.fill_none(min_dr, np.inf) # fills events with no jets (min_dr=None) with inf so they fail the dr_cut

# Require a valid match within dr_cut
has_match = (min_dr_filled < dr_cut)

# Keep only matched soft muons and their matched jets
soft_muon_in_goodjet = soft_muon[has_match]
c_Jets = jets[best_jet_idx]
c_Jets = c_Jets[has_match]

# Select events with atleast one c-jet:
nC_Jets = dak.num(c_Jets, axis=1)
evt_has_cJet = (nC_Jets >= 1)
c_Jets = c_Jets[evt_has_cJet]

# Total number of cJets in the dataset:
nC_Jets = dak.num(c_Jets, axis=1)
total_cJets = dak.sum(nC_Jets)
cutflow["3. c-Jets after cuts"] = total_cJets
print("Total number of c-jets in the dataset:", total_cJets.compute())

Defining charm Selection ...
Total number of soft muons: 1960
Total number of jets eligible for matching: 209480
Total number of c-jets in the dataset: 630


## 7. Signal Extraction:

We want exactly one W boson candidate and at least one c-jet candidate.

In [ ]:
# 

## 8. Plots: